# Basic network statistics: six company networks

## 1. Load the data
Loads each company as an undirected graph. Duplicate friendships (A,B and B,A) become one edge; self-loops are removed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

NETWORKS = ['S1', 'S2', 'M1', 'M2', 'L1', 'L2']
DATA = Path('data') if Path('data').exists() else Path('../data')


def load_network(name):
    '''Reads nodes.csv and edges.csv of one company into an undirected graph without duplicates or self-loops.'''
    folder = DATA / f'{name}.csv'
    nodes = pd.read_csv(folder / 'nodes.csv', skipinitialspace=True)
    edges = pd.read_csv(folder / 'edges.csv', skipinitialspace=True)
    G = nx.Graph()
    G.add_nodes_from(nodes.iloc[:, 0])
    G.add_edges_from(zip(edges.iloc[:, 0], edges.iloc[:, 1])) # here is where we drop duplicates (m1 and l1)
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


graphs = {name: load_network(name) for name in NETWORKS}

## 2. Nodes, edges, average degree and density
Average degree = 2E / N. Density = share of all possible pairs that are connected.

In [ ]:
def basic_stats(G):
    '''Number of nodes and edges, average degree and density of one graph.'''
    N, E = G.number_of_nodes(), G.number_of_edges()
    return {'nodes': N, 'edges': E, 'average degree': 2 * E / N, 'density': nx.density(G)}


basic = pd.DataFrame.from_dict({name: basic_stats(G) for name, G in graphs.items()}, orient='index')
basic.round(4)

## 3. Degree distribution
Share of nodes with at least k friends (CCDF), on log–log axes.

In [ ]:
def degree_summary(G):
    '''Minimum, median and maximum degree of one graph.'''
    k = [d for _, d in G.degree()]
    return {'min degree': min(k), 'median degree': np.median(k), 'max degree': max(k)}


def degree_ccdf(G):
    '''Each degree value k and the share of nodes with degree >= k.'''
    k = np.array([d for _, d in G.degree()])
    values = np.unique(k)
    return values, [(k >= v).mean() for v in values]


fig, ax = plt.subplots(figsize=(7, 4.5))
for name, G in graphs.items():
    k, share = degree_ccdf(G)
    ax.loglog(k, share, label=name)
ax.set_xlabel('degree k')
ax.set_ylabel('P(K ≥ k)')
ax.set_title('Degree distribution (CCDF)')
ax.legend()
plt.show()

pd.DataFrame.from_dict({name: degree_summary(G) for name, G in graphs.items()}, orient='index')

## 4. Clustering
Transitivity = share of closed triangles in the whole network. Average clustering = mean of every node's local clustering.

In [ ]:
def clustering_stats(G):
    '''Global (transitivity) and average local clustering of one graph.'''
    return {'transitivity': nx.transitivity(G), 'average clustering': nx.average_clustering(G)}


clustering = pd.DataFrame.from_dict({name: clustering_stats(G) for name, G in graphs.items()}, orient='index')
clustering.round(3)

## 5. Centrality
Degree, closeness, betweenness and eigenvector centrality for every node. On networks above 2,000 nodes, betweenness is estimated from 500 sampled nodes to keep it fast. This cell takes several minutes.

In [ ]:
def centralities(G):
    '''Table with four centrality scores for every node of one graph.'''
    k = None if G.number_of_nodes() <= 2000 else 500
    return pd.DataFrame({
        'degree': nx.degree_centrality(G),
        'closeness': nx.closeness_centrality(G),
        'betweenness': nx.betweenness_centrality(G, k=k, seed=42),
        'eigenvector': nx.eigenvector_centrality(G, max_iter=1000),
    })


cent = {}
for name, G in graphs.items():
    cent[name] = centralities(G)
    print(name, 'done')

In [ ]:
def max_centralities(df):
    '''Highest score of each centrality measure in one network.'''
    return {f'max {m}': df[m].max() for m in df.columns}


pd.DataFrame.from_dict({name: max_centralities(df) for name, df in cent.items()}, orient='index').round(3)

In [ ]:
def top_nodes(name, n=5):
    '''The n highest-scoring nodes for each measure in one network.'''
    df = cent[name]
    return pd.DataFrame({m: df[m].nlargest(n).index for m in df.columns})


top_nodes('L2')

## 6. Connected components
Number of separate pieces of the network and the share of nodes in the largest one.


In [5]:
# we added this just to make sure the whole network is one piece (connected)
def component_stats(G):
    '''Number of connected components and the share of nodes in the largest one.'''
    largest = max(nx.connected_components(G), key=len)
    return {'components': nx.number_connected_components(G),
            'largest component share': len(largest) / G.number_of_nodes()}


pd.DataFrame.from_dict({name: component_stats(G) for name, G in graphs.items()}, orient='index')

,components,largest component share
S1,1,1.0
S2,1,1.0
M1,1,1.0
M2,1,1.0
L1,1,1.0
L2,1,1.0


## 7. Path length and diameter
Average number of steps between two nodes, and the longest shortest path. Takes a few minutes on the large networks.

In [3]:
def path_stats(G):
    '''Average shortest path length and diameter, from one breadth-first search per node.'''
    total, pairs, diameter = 0, 0, 0
    for node in G:
        dist = nx.single_source_shortest_path_length(G, node)
        total += sum(dist.values())
        pairs += len(dist) - 1
        diameter = max(diameter, max(dist.values()))
    return {'average path length': total / pairs, 'diameter': diameter}


paths = {}
for name, G in graphs.items():
    paths[name] = path_stats(G)
    print(name, 'done')
pd.DataFrame.from_dict(paths, orient='index').round(2)

S1 done
S2 done
M1 done
M2 done
L1 done
L2 done


,average path length,diameter
S1,2.71,7
S2,2.88,6
M1,3.04,7
M2,2.76,5
L1,6.26,16
L2,3.50,9
